In [1]:
# Import libraries
import pandas as pd
import numpy as np
from datetime import timedelta
import warnings
warnings.filterwarnings('ignore')

In [2]:
# LOAD CLEANED DATA
print("\n LOADING CLEANED DATA...")
print("-" * 80)

df = pd.read_csv('../data/processed/cleaned_data.csv')
df['date'] = pd.to_datetime(df['date'])
df['dob'] = pd.to_datetime(df['dob'])

print(f"✓ Loaded {len(df):,} records for {df['child_id'].nunique():,} children")


 LOADING CLEANED DATA...
--------------------------------------------------------------------------------
✓ Loaded 310,850 records for 57,684 children


In [3]:
# AGE-BASED FEATURES
print("\n CREATING AGE-BASED FEATURES")
print("-" * 80)

# Recalculate age in different units
df['age_days'] = (df['date'] - df['dob']).dt.days
df['age_weeks'] = df['age_days'] / 7
df['age_months'] = df['age_days'] / 30.44
df['age_years'] = df['age_days'] / 365.25

print("✓ Created age features: days, weeks, months, years")

# Age categories
def categorize_age(age_months):
    if age_months < 6:
        return '0-6m'
    elif age_months < 12:
        return '6-12m'
    elif age_months < 24:
        return '12-24m'
    elif age_months < 36:
        return '24-36m'
    else:
        return '36m+'

df['age_category'] = df['age_months'].apply(categorize_age)
print("✓ Created age categories")


 CREATING AGE-BASED FEATURES
--------------------------------------------------------------------------------
✓ Created age features: days, weeks, months, years
✓ Created age categories


In [4]:
# GROWTH VELOCITY FEATURES (PER CHILD)
print("\n CREATING GROWTH VELOCITY FEATURES")
print("-" * 80)

# Sort by child and date
df = df.sort_values(['child_id', 'date']).reset_index(drop=True)

# Calculate changes from previous measurement
df['height_prev'] = df.groupby('child_id')['height'].shift(1)
df['weight_prev'] = df.groupby('child_id')['weight'].shift(1)
df['date_prev'] = df.groupby('child_id')['date'].shift(1)

# Calculate time difference in days
df['days_since_prev'] = (df['date'] - df['date_prev']).dt.days

# Calculate growth velocities (per day)
df['height_velocity_daily'] = (df['height'] - df['height_prev']) / df['days_since_prev']
df['weight_velocity_daily'] = (df['weight'] - df['weight_prev']) / df['days_since_prev']

# Calculate growth velocities (per month - more interpretable)
df['height_velocity_monthly'] = df['height_velocity_daily'] * 30.44
df['weight_velocity_monthly'] = df['weight_velocity_daily'] * 30.44

# Handle first measurement
df['height_velocity_monthly'].fillna(0, inplace=True)
df['weight_velocity_monthly'].fillna(0, inplace=True)

print("✓ Created growth velocity features")
print(f"  - Height velocity (cm/month)")
print(f"  - Weight velocity (kg/month)")


 CREATING GROWTH VELOCITY FEATURES
--------------------------------------------------------------------------------
✓ Created growth velocity features
  - Height velocity (cm/month)
  - Weight velocity (kg/month)


In [5]:
# CUMULATIVE GROWTH FEATURES
print("\n CREATING CUMULATIVE GROWTH FEATURES")
print("-" * 80)

# Total growth from first measurement
df['height_first'] = df.groupby('child_id')['height'].transform('first')
df['weight_first'] = df.groupby('child_id')['weight'].transform('first')

df['height_growth_total'] = df['height'] - df['height_first']
df['weight_growth_total'] = df['weight'] - df['weight_first']

print("✓ Created cumulative growth since first measurement")



 CREATING CUMULATIVE GROWTH FEATURES
--------------------------------------------------------------------------------
✓ Created cumulative growth since first measurement


In [6]:
# MEASUREMENT PATTERN FEATURES
print("\n CREATING MEASUREMENT PATTERN FEATURES")
print("-" * 80)

# Number of measurements so far for each child
df['num_measurements_so_far'] = df.groupby('child_id').cumcount() + 1

# Days since first measurement
df['date_first'] = df.groupby('child_id')['date'].transform('first')
df['days_since_first_measurement'] = (df['date'] - df['date_first']).dt.days

# Average measurement interval for child
df['avg_measurement_interval'] = df.groupby('child_id')['days_since_prev'].transform('mean')

print("✓ Created measurement pattern features")


 CREATING MEASUREMENT PATTERN FEATURES
--------------------------------------------------------------------------------
✓ Created measurement pattern features


In [7]:
# Z-SCORE CHANGE FEATURES
print("\n CREATING Z-SCORE CHANGE FEATURES")
print("-" * 80)

z_score_cols = ['zlen', 'zwei', 'zwfl', 'zbmi']

for col in z_score_cols:
    df[f'{col}_prev'] = df.groupby('child_id')[col].shift(1)
    df[f'{col}_change'] = df[col] - df[f'{col}_prev']
    df[f'{col}_change'].fillna(0, inplace=True)

print("✓ Created z-score change features for all WHO metrics")


 CREATING Z-SCORE CHANGE FEATURES
--------------------------------------------------------------------------------
✓ Created z-score change features for all WHO metrics


In [8]:
# GENDER-SPECIFIC FEATURES
print("\n CREATING GENDER-SPECIFIC FEATURES")
print("-" * 80)

# Encode gender as numeric
df['gender_numeric'] = (df['gender'] == 'M').astype(int)

print("✓ Created gender encoding (0=Female, 1=Male)")



 CREATING GENDER-SPECIFIC FEATURES
--------------------------------------------------------------------------------
✓ Created gender encoding (0=Female, 1=Male)


In [9]:
# SEASONAL/TEMPORAL FEATURES
print("\n CREATING SEASONAL/TEMPORAL FEATURES")
print("-" * 80)

# Extract temporal components
df['measurement_month'] = df['date'].dt.month
df['measurement_quarter'] = df['date'].dt.quarter
df['measurement_year'] = df['date'].dt.year

# Cyclical encoding for month
df['month_sin'] = np.sin(2 * np.pi * df['measurement_month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['measurement_month'] / 12)

print("✓ Created temporal features (month, quarter, year)")
print("✓ Created cyclical encoding for seasonality")



 CREATING SEASONAL/TEMPORAL FEATURES
--------------------------------------------------------------------------------
✓ Created temporal features (month, quarter, year)
✓ Created cyclical encoding for seasonality


In [10]:
# GROWTH PATTERN INDICATORS
print("\n CREATING GROWTH PATTERN INDICATORS")
print("-" * 80)

df['is_stunted'] = (df['zlen'] < -2).astype(int)

df['is_underweight'] = (df['zwei'] < -2).astype(int)

df['is_wasted'] = (df['zwfl'] < -2).astype(int)

df['is_overweight'] = (df['zbmi'] > 2).astype(int)

print("✓ Created WHO malnutrition indicators")
print(f"  - Stunted: {df['is_stunted'].sum():,} ({df['is_stunted'].mean()*100:.1f}%)")
print(f"  - Underweight: {df['is_underweight'].sum():,} ({df['is_underweight'].mean()*100:.1f}%)")
print(f"  - Wasted: {df['is_wasted'].sum():,} ({df['is_wasted'].mean()*100:.1f}%)")


 CREATING GROWTH PATTERN INDICATORS
--------------------------------------------------------------------------------
✓ Created WHO malnutrition indicators
  - Stunted: 128,772 (41.4%)
  - Underweight: 69,064 (22.2%)
  - Wasted: 29,694 (9.6%)


In [11]:
# RATIO AND PROPORTION FEATURES
print("\n CREATING RATIO FEATURES")
print("-" * 80)

# Weight-to-height ratio
df['weight_height_ratio'] = df['weight'] / df['height']

# Ponderal Index (weight/height³)
df['ponderal_index'] = df['weight'] / (df['height'] ** 3) * 100

print("✓ Created ratio features")


 CREATING RATIO FEATURES
--------------------------------------------------------------------------------
✓ Created ratio features


In [12]:
# CLEAN UP AND SELECT FINAL FEATURES
print("\n SELECTING FINAL FEATURE SET")
print("-" * 80)

# Drop intermediate/temporary columns
columns_to_drop = [
    'height_prev', 'weight_prev', 'date_prev', 'days_since_prev',
    'height_first', 'weight_first', 'date_first',
    'zlen_prev', 'zwei_prev', 'zwfl_prev', 'zbmi_prev',
    'height_velocity_daily', 'weight_velocity_daily'
]

df_features = df.drop(columns=columns_to_drop, errors='ignore')

print(f"✓ Final dataset has {len(df_features.columns)} features")


 SELECTING FINAL FEATURE SET
--------------------------------------------------------------------------------
✓ Final dataset has 46 features


In [14]:
# SAVE ENGINEERED DATASET
print("\n SAVING ENGINEERED DATASET")
print("-" * 80)

# Save full feature set
df_features.to_csv('../data/processed/features_engineered.csv', index=False)
print("✓ Saved to: ../data/processed/features_engineered.csv")

# Save feature list with descriptions
feature_descriptions = {
    'Identifiers': ['child_id', 'hh_id'],
    'Demographics': ['gender', 'gender_numeric', 'dob'],
    'Geography': ['district', 'upazila', 'union', 'village'],
    'Raw Measurements': ['date', 'height', 'weight', 'cbmi'],
    'Age Features': ['age_days', 'age_weeks', 'age_months', 'age_years', 'age_category'],
    'Z-Scores': ['zlen', 'zwei', 'zwfl', 'zbmi'],
    'Z-Score Changes': ['zlen_change', 'zwei_change', 'zwfl_change', 'zbmi_change'],
    'Growth Velocities': ['height_velocity_monthly', 'weight_velocity_monthly'],
    'Cumulative Growth': ['height_growth_total', 'weight_growth_total'],
    'Measurement Patterns': ['measurement_number', 'num_measurements_so_far', 
                             'days_since_first_measurement', 'avg_measurement_interval'],
    'Temporal': ['measurement_month', 'measurement_quarter', 'measurement_year',
                 'month_sin', 'month_cos'],
    'Malnutrition Indicators': ['is_stunted', 'is_underweight', 'is_wasted', 'is_overweight'],
    'Ratios': ['weight_height_ratio', 'ponderal_index'],
    'Quality': ['has_who_flag']
}

with open('../data/processed/feature_descriptions.txt', 'w') as f:
    f.write("FEATURE CATEGORIES\n")
    f.write("=" * 80 + "\n\n")
    for category, features in feature_descriptions.items():
        f.write(f"{category}:\n")
        for feature in features:
            f.write(f"  - {feature}\n")
        f.write("\n")

print("✓ Saved feature descriptions to: ../data/processed/feature_descriptions.txt")


 SAVING ENGINEERED DATASET
--------------------------------------------------------------------------------
✓ Saved to: ../data/processed/features_engineered.csv
✓ Saved feature descriptions to: ../data/processed/feature_descriptions.txt


In [15]:
# FEATURE ENGINEERING SUMMARY
print("\n" + "=" * 80)
print("FEATURE ENGINEERING SUMMARY")
print("=" * 80)

print(f"\nTotal Features: {len(df_features.columns)}")
print(f"Total Records: {len(df_features):,}")
print(f"Total Children: {df_features['child_id'].nunique():,}")

print("\nFeature Categories:")
for category, features in feature_descriptions.items():
    print(f"  {category}: {len(features)} features")

print(f"\nAge Range:")
print(f"  Min: {df_features['age_months'].min():.1f} months")
print(f"  Max: {df_features['age_months'].max():.1f} months")
print(f"  Mean: {df_features['age_months'].mean():.1f} months")

print(f"\nGrowth Velocity Statistics:")
print(f"  Height velocity: {df_features['height_velocity_monthly'].mean():.3f} ± {df_features['height_velocity_monthly'].std():.3f} cm/month")
print(f"  Weight velocity: {df_features['weight_velocity_monthly'].mean():.3f} ± {df_features['weight_velocity_monthly'].std():.3f} kg/month")


FEATURE ENGINEERING SUMMARY

Total Features: 46
Total Records: 310,850
Total Children: 57,684

Feature Categories:
  Identifiers: 2 features
  Demographics: 3 features
  Geography: 4 features
  Raw Measurements: 4 features
  Age Features: 5 features
  Z-Scores: 4 features
  Z-Score Changes: 4 features
  Growth Velocities: 2 features
  Cumulative Growth: 2 features
  Measurement Patterns: 4 features
  Temporal: 5 features
  Malnutrition Indicators: 4 features
  Ratios: 2 features
  Quality: 1 features

Age Range:
  Min: 0.0 months
  Max: 93.0 months
  Mean: 35.5 months

Growth Velocity Statistics:
  Height velocity: 0.627 ± 1.959 cm/month
  Weight velocity: 0.126 ± 1.036 kg/month
